# Exploratory Data Analysis of Features.xlsx\n
\n
This notebook provides an exploratory data analysis of the `features.xlsx` file, which is generated by the data pipeline after fetching and engineering features. The goal is to understand the quality, completeness, and structure of the data available for model training and next-quarter predictions.\n
\n
It will help us identify:\n
- Overall dataset characteristics.\n
- Number of unique tickers and their data coverage.\n
- Missing values, especially for `next_quarter_return` (which indicates data points for future predictions).\n
- Distribution of engineered features.\n
- Data points per ticker.

In [ ]:
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
import os\n
import sys\n
\n
# Add the parent directory to the Python path to import from src.config\n
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))\n
from src.config import FEATURES_PATH\n
\n
# Set display options for pandas DataFrames\n
pd.set_option('display.max_columns', None)\n
pd.set_option('display.max_rows', 100)\n
pd.set_option('display.width', 1000)\n
\n
print(f"Attempting to load features from: {FEATURES_PATH}")

In [ ]:
# Load the features dataset\n
try:\n
    features_df = pd.read_excel(FEATURES_PATH)\n
    print("Features dataset loaded successfully.")\n
except FileNotFoundError:\n
    print(f"Error: {FEATURES_PATH} not found. Please ensure the pipeline has been run to generate this file.")\n
    features_df = pd.DataFrame() # Create an empty DataFrame to prevent further errors\n
except Exception as e:\n
    print(f"An error occurred while loading features.xlsx: {e}")\n
    features_df = pd.DataFrame()

## 1. Dataset Overview

In [ ]:
if not features_df.empty:\n
    print("### DataFrame Head ###")\n
    display(features_df.head())\n
\n
    print("\n### DataFrame Info ###")\n
    features_df.info()\n
\n
    print("\n### DataFrame Description (Numerical Features) ###")\n
    display(features_df.describe())

## 2. Ticker and Date Information

In [ ]:
if not features_df.empty:\n
    print(f"Number of unique tickers: {features_df['ticker'].nunique()}")\n
    print(f"Unique tickers: {features_df['ticker'].unique().tolist()}")\n
\n
    features_df['filing_date'] = pd.to_datetime(features_df['filing_date'])\n
    print(f"\nFiling dates range from {features_df['filing_date'].min().strftime('%Y-%m-%d')} to {features_df['filing_date'].max().strftime('%Y-%m-%d')}")\n
\n
    print("\nNumber of data points per ticker:")\n
    display(features_df['ticker'].value_counts().sort_index())

## 3. Missing Values Analysis

In [ ]:
if not features_df.empty:\n
    print("### Missing values per column ###")\n
    missing_values = features_df.isnull().sum()\n
    display(missing_values[missing_values > 0].sort_values(ascending=False))\n
\n
    # Identify rows where 'next_quarter_return' is NaN (these are candidates for prediction)\n
    nan_next_quarter_return_df = features_df[features_df['next_quarter_return'].isna()].copy()\n
    print("\n### Data points with NaN 'next_quarter_return' (candidates for next quarter prediction) ###")\n
    if not nan_next_quarter_return_df.empty:\n
        display(nan_next_quarter_return_df[['ticker', 'filing_date']].sort_values(by=['ticker', 'filing_date']))\n
        print(f"Number of tickers with NaN 'next_quarter_return': {nan_next_quarter_return_df['ticker'].nunique()}")\n
        print(f"Tickers with NaN 'next_quarter_return': {nan_next_quarter_return_df['ticker'].unique().tolist()}")\n
    else:\n
        print("No data points with NaN 'next_quarter_return' found. This might indicate that all historical data has known outcomes, or that latest filings were dropped during feature engineering.")

## 4. Feature Distributions

In [ ]:
if not features_df.empty:\n
    # Select numerical features for plotting\n
    numerical_features = ['revenue_growth', 'net_margin', 'sentiment_score', 'sentiment_change', 'next_quarter_return']\n
    \n
    plt.figure(figsize=(15, 10))\n
    for i, feature in enumerate(numerical_features):\n
        if feature in features_df.columns:\n
            plt.subplot(2, 3, i + 1) # Adjust subplot grid as needed\n
            sns.histplot(features_df[feature].dropna(), kde=True)\n
            plt.title(f'Distribution of {feature}')\n
    plt.tight_layout()\n
    plt.show()\n
\n
    # Correlation matrix\n
    print("\n### Correlation Matrix of Numerical Features ###")\n
    correlation_matrix = features_df[numerical_features].corr()\n
    plt.figure(figsize=(8, 6))\n
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")\n
    plt.title('Correlation Matrix')\n
    plt.show()

## Summary of Findings\n
\n
Based on the analysis above, you should be able to identify:\n
- Whether `features.xlsx` is being generated correctly with all expected tickers.\n
- If there are enough data points (`train_data_ticker`) per ticker to train individual models (a minimum of 5-10 is generally recommended, depending on the model).\n
- Which tickers have `next_quarter_return` as `NaN`, indicating they are the latest data points that the `generate_next_quarter_prediction` function in `src/model.py` should target.\n
- Any unexpected `NaN` values in critical feature columns that might prevent model training or prediction for certain tickers.